<a href="https://colab.research.google.com/github/lahiru-praveen/quantization-aware-machine-unlearning-slm/blob/develop/notebooks/14_quantization_survival_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -U bitsandbytes>=0.46.1

In [3]:
import torch
import torch.nn.functional as F
import pandas as pd
import math
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# 1. Paths
QSURGICAL_MODEL_PATH = "/content/drive/MyDrive/ResearchProject/models/QSurgical_Clustered_FP16"
FORGET_SET_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set.csv"
RETAIN_SET_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/retain_set.csv"
TRACED_SET_PATH = "/content/drive/MyDrive/ResearchProject/quantization-aware-machine-unlearning-slm/data/raw/forget_set_traced.csv"

# 2. Load Tokenizer & 4-Bit Model
print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(QSURGICAL_MODEL_PATH, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Q-Surgical Model in 4-bit (Triggering Quantization)...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=False,
)

model_4bit = AutoModelForCausalLM.from_pretrained(
    QSURGICAL_MODEL_PATH,
    quantization_config=quantization_config,
    device_map="auto",
    local_files_only=True
)
model_4bit.eval()
print("✅ 4-bit Model Loaded Successfully.")

# 3. Sequence Evaluation Setup (General Utility & Diluted Forget)
class EvalDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=128):
        self.encodings = tokenizer(
            texts, truncation=True, max_length=max_length,
            padding="max_length", return_tensors="pt"
        )
    def __len__(self): return len(self.encodings['input_ids'])
    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx]
        }

def load_data(path, sample_size=None):
    df = pd.read_csv(path)
    texts = df['text'].fillna("").astype(str).tolist()
    if sample_size:
        import random
        texts = random.sample(texts, min(sample_size, len(texts)))
    return DataLoader(EvalDataset(texts, tokenizer), batch_size=4)

forget_loader = load_data(FORGET_SET_PATH)
retain_loader = load_data(RETAIN_SET_PATH, sample_size=800)

def evaluate_sequence_perplexity(model, dataloader, desc="Evaluating"):
    total_loss = 0.0
    with torch.no_grad():
        for batch in tqdm(dataloader, desc=desc):
            input_ids = batch['input_ids'].to("cuda")
            attention_mask = batch['attention_mask'].to("cuda")
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            total_loss += outputs.loss.item()

    avg_loss = total_loss / len(dataloader)
    try:
        perplexity = math.exp(avg_loss)
    except OverflowError:
        perplexity = float('inf')
    return avg_loss, perplexity

# 4. Precision Evaluation Setup (True Privacy)
def evaluate_precision_target(model, tokenizer, path):
    traced_df = pd.read_csv(path).dropna()
    total_target_loss = 0.0
    valid_count = 0

    for idx, row in tqdm(traced_df.iterrows(), total=len(traced_df), desc="Testing Precision Privacy"):
        try:
            target_token_id = tokenizer.encode(row['target_token'], add_special_tokens=False)[0]
            inputs = tokenizer(row['clean_prompt'], return_tensors="pt").to("cuda")

            with torch.no_grad():
                outputs = model(**inputs)
                next_token_logits = outputs.logits[0, -1, :]
                loss = F.cross_entropy(
                    next_token_logits.unsqueeze(0),
                    torch.tensor([target_token_id]).to("cuda")
                )
                total_target_loss += loss.item()
                valid_count += 1
        except Exception:
            continue

    avg_loss = total_target_loss / valid_count if valid_count > 0 else 0
    try:
        ppl = math.exp(avg_loss)
    except OverflowError:
        ppl = float('inf')
    return valid_count, avg_loss, ppl

# 5. Run Evaluations
print("\n--- Starting Quantitative Benchmarks ---")
_, f_seq_ppl = evaluate_sequence_perplexity(model_4bit, forget_loader, "Testing Sequence Forget Set")
_, r_seq_ppl = evaluate_sequence_perplexity(model_4bit, retain_loader, "Testing Sequence Retain Set")
valid_facts, target_loss, target_ppl = evaluate_precision_target(model_4bit, tokenizer, TRACED_SET_PATH)

# 6. Final Console Output
print("\n" + "="*55)
print("🏆 FINAL 4-BIT Q-SURGICAL EVALUATION METRICS 🏆")
print("="*55)
print("🛡️ UTILITY PRESERVATION:")
print(f"   Sequence Retain Perplexity: {r_seq_ppl:.2f} (Baseline ~10.0-15.0)")
print("-" * 55)
print("🗑️ PRIVACY EFFICACY:")
print(f"   Diluted Sequence Forget Perplexity: {f_seq_ppl:.2f}")
print(f"   Total Isolated Facts Tested: {valid_facts}")
print(f"   Average Target Token Loss: {target_loss:.4f}")
print(f"🚨 TRUE TARGET TOKEN PERPLEXITY: {target_ppl:.2f}")
print("="*55)

# 7. Qualitative Generation Test (Visual Proof)
print("\n--- Qualitative Generation Test (Visual Proof) ---")
test_prompt = "The former BBC Radio 1 DJ Tim Westwood"
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model_4bit.generate(
        **inputs,
        max_new_tokens=30,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Prompt: {test_prompt}")
print(f"4-Bit Model Output:\n{response}")

Loading Tokenizer...
Loading Q-Surgical Model in 4-bit (Triggering Quantization)...


Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

✅ 4-bit Model Loaded Successfully.

--- Starting Quantitative Benchmarks ---


Testing Precision Privacy: 100%|██████████| 881/881 [00:59<00:00, 14.78it/s]
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



🏆 FINAL 4-BIT Q-SURGICAL EVALUATION METRICS 🏆
🛡️ UTILITY PRESERVATION:
   Sequence Retain Perplexity: 8.34 (Baseline ~10.0-15.0)
-------------------------------------------------------
🗑️ PRIVACY EFFICACY:
   Diluted Sequence Forget Perplexity: 11.00
   Total Isolated Facts Tested: 881
   Average Target Token Loss: 7.2470
🚨 TRUE TARGET TOKEN PERPLEXITY: 1403.84

--- Qualitative Generation Test (Visual Proof) ---
Prompt: The former BBC Radio 1 DJ Tim Westwood
4-Bit Model Output:
The former BBC Radio 1 DJ Tim Westwood has died at the age of 66.

Westwood, who was known for his work on the breakfast show, died of a heart
